In [1]:
import random

import numpy as np
import torch
from riichienv import RiichiEnv
from riichienv.visualizer import GameViewer

import sys
sys.path.append("riichienv-ml/src")

from riichienv_ml.config import load_config, import_class
from riichienv_ml.utils import resolve_train_device

In [2]:
CONFIG_PATH = "riichienv-ml/src/riichienv_ml/configs/4p/ppo.yml"
MODEL_PATH = "checkpoints/model_50.pth"

HERO_SEAT = 0
EPISODES = 1
TOPK = 5
SAMPLE = False
TEMPERATURE = 1.0
SEED = 42

In [3]:
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

cfg_root = load_config(CONFIG_PATH)
cfg = cfg_root.ppo
game = cfg.game

device = torch.device(resolve_train_device(cfg.device))

ModelClass = import_class(cfg.model_class)
EncoderClass = import_class(cfg.encoder_class)

model = ModelClass(**cfg.model.model_dump()).to(device)
encoder = EncoderClass(tile_dim=game.tile_dim)

state = torch.load(MODEL_PATH, map_location=device)
if "state_dict" in state:
    state = state["state_dict"]

model.load_state_dict(state, strict=False)
model.eval()

print("Loaded model on", device)

2026-03-04 22:01:24.676 | WARNING  | riichienv_ml.utils:resolve_train_device:167 - CUDA is unavailable. Falling back to MPS on macOS.


Loaded model on mps


In [4]:
def choose_action(model, encoder, obs):
    feat = encoder.encode(obs).unsqueeze(0).to(device)
    mask = torch.from_numpy(np.frombuffer(obs.mask(), dtype=np.uint8).copy()).to(device).bool()

    with torch.no_grad():
        logits = model(feat)
        if isinstance(logits, tuple):
            logits = logits[0][0]
        else:
            logits = logits[0]

        logits = logits.masked_fill(~mask, -1e9)

        if SAMPLE:
            probs = torch.softmax(logits / TEMPERATURE, dim=-1)
            action_idx = torch.multinomial(probs, 1).item()
        else:
            action_idx = torch.argmax(logits).item()

    return action_idx, logits.cpu()

In [5]:
env = RiichiEnv(game_mode=game.game_mode)

obs_dict = env.reset(
    scores=list(game.starting_scores),
    round_wind=0,
    oya=0,
    honba=0,
    kyotaku=0,
)

hero_decisions = 0

while not env.done():
    actions = {}

    for pid, obs in obs_dict.items():

        legal = obs.legal_actions()
        if not legal:
            continue

        if pid == HERO_SEAT:

            action_idx, values = choose_action(model, encoder, obs)
            action = obs.find_action(action_idx)

            if action is None:
                action = legal[0]

            hero_decisions += 1

            print(
                f"decision={hero_decisions} choose={action_idx} action={action}"
            )

            actions[pid] = action

        else:
            actions[pid] = random.choice(legal)

    obs_dict = env.step(actions)

decision=1 choose=14 action=Action(action_type=Discard, tile=Some(58), consume_tiles=[], actor=Some(0))
decision=2 choose=32 action=Action(action_type=Discard, tile=Some(131), consume_tiles=[], actor=Some(0))
decision=3 choose=0 action=Action(action_type=Discard, tile=Some(1), consume_tiles=[], actor=Some(0))
decision=4 choose=3 action=Action(action_type=Discard, tile=Some(15), consume_tiles=[], actor=Some(0))
decision=5 choose=16 action=Action(action_type=Discard, tile=Some(67), consume_tiles=[], actor=Some(0))
decision=6 choose=17 action=Action(action_type=Discard, tile=Some(70), consume_tiles=[], actor=Some(0))
decision=7 choose=1 action=Action(action_type=Discard, tile=Some(6), consume_tiles=[], actor=Some(0))
decision=8 choose=41 action=Action(action_type=Pon, tile=Some(122), consume_tiles=[120, 121], actor=Some(0))
decision=9 choose=20 action=Action(action_type=Discard, tile=Some(83), consume_tiles=[], actor=Some(0))
decision=10 choose=18 action=Action(action_type=Discard, tile=S

In [6]:
print("scores:", list(env.scores()))
print("ranks:", list(env.ranks()))

scores: [73200, 6400, 4400, 16000]
ranks: [1, 3, 4, 2]


In [7]:
viewer = GameViewer.from_env(env, perspective=HERO_SEAT)
viewer